## Transformer中的Encoder

In [3]:
import torch

batch_size = 2
seq_len = 5
d_model = 512
vocab_size = 100

inputs = torch.randint(0, vocab_size, (batch_size, seq_len))
inputs

tensor([[55, 75, 23, 27, 15],
        [56, 97, 11, 89, 12]])

In [4]:
embedding = torch.nn.Embedding(vocab_size, d_model)
outputs = embedding(inputs)
outputs.shape

torch.Size([2, 5, 512])

In [5]:
import math

pe = torch.zeros(seq_len, d_model)
position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
pe[:, 0::2] = torch.sin(position * div_term)
pe[:, 1::2] = torch.cos(position * div_term)
pe

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  8.2186e-01,  ...,  1.0000e+00,
          1.0366e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.3641e-01,  ...,  1.0000e+00,
          2.0733e-04,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.4509e-01,  ...,  1.0000e+00,
          3.1099e-04,  1.0000e+00],
        [-7.5680e-01, -6.5364e-01, -6.5717e-01,  ...,  1.0000e+00,
          4.1465e-04,  1.0000e+00]])

In [6]:
encoder_inputs = outputs + pe
encoder_inputs.shape

torch.Size([2, 5, 512])

In [ ]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        q_k = k.size(-1)
        attn_score = attn_score / torch.sqrt(torch.tensor(q_k))
        attn_weight = torch.softmax(attn_score, dim=-1)

        o = torch.matmul(attn_weight, v)
        attn_out = o.transpose(1, 2).reshape(batch_size, seq_len, self.attn_dim)
        return self.out_proj(attn_out)

In [ ]:
attn = MultiHeadAttention(d_model, d_model, d_model, 8)
encoder_attn_outputs = attn(encoder_inputs)
encoder_attn_outputs.shape

In [ ]:
encoder_attn_add_outputs = encoder_attn_outputs + encoder_inputs
encoder_attn_add_outputs.shape

In [ ]:
layer_norm = nn.LayerNorm(d_model)
encoder_attn_add_outputs = layer_norm(encoder_attn_add_outputs)
encoder_attn_add_outputs.shape

In [ ]:
encoder_attn_add_outputs[0][0]

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))


feed_forward = FeedForward(d_model, 2048)
feed_forward_outputs = feed_forward(encoder_attn_add_outputs)
feed_forward_outputs.shape

In [ ]:
feed_forward_add_outputs = feed_forward_outputs + encoder_attn_add_outputs
feed_forward_add_outputs.shape

In [ ]:
feed_forward_add_outputs[0][0]

In [ ]:
encoder_outputs = layer_norm(feed_forward_add_outputs)
encoder_outputs.shape

### 手写encoder

In [8]:
import torch.nn as nn
import torch

class Encoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, d_ff, n_heads) for _ in range(n_encoder_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm(x + self.attn(x))
        x = self.norm(x + self.ffn(x))
        return x

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        q_k = k.size(-1)
        attn_score = attn_score / torch.sqrt(torch.tensor(q_k))
        attn_weight = torch.softmax(attn_score, dim=-1)

        o = torch.matmul(attn_weight, v)
        attn_out = o.transpose(1, 2).reshape(batch_size, seq_len, self.attn_dim)
        return self.out_proj(attn_out)

In [9]:
encoder = Encoder(512, 2048, 8, 6)

encoder_outputs = encoder(encoder_inputs)
encoder_outputs.shape

torch.Size([2, 5, 512])